# 12-03 TabPFN Final Test mit drei Targets

Dieses Notebook fuehrt den echten Out-of-Sample-Test auf 2024/2025 aus.

Es trainiert oder laedt pro Variante drei finale Classifier:
- `Top5`
- `Top10`
- `Top20`

Der einzige finale Ranking-Score ist:

```python
score_topk_raw_sum = p_top5_raw + p_top10_raw + p_top20_raw
```

Zusaetzlich wird pro Fahrer die wahrscheinlichste disjunkte Klasse bestimmt.


In [1]:
from pathlib import Path
from time import perf_counter
import json
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "model_data").exists() and (candidate / "src" / "Notebooks").exists():
            return candidate
    raise FileNotFoundError("Projektwurzel mit data/model_data und src/Notebooks nicht gefunden.")

PROJECT_ROOT = find_project_root()
MODEL_DATA_DIR = PROJECT_ROOT / "data" / "model_data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABPFN_DIR = PROCESSED_DIR / "tabpfn"
LEGACY_DIR = PROCESSED_DIR / "tabpfn_3_experiments"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"MODEL_DATA_DIR: {MODEL_DATA_DIR}")


PROJECT_ROOT: /Users/roberthendrich/GADA-Group3-Cycling-Stage-Prediction
MODEL_DATA_DIR: /Users/roberthendrich/GADA-Group3-Cycling-Stage-Prediction/data/model_data


In [2]:
FEATURE_COLUMNS = [
    "distance",
    "vertical_meters",
    "stage_nr",
    "team_tier",
    "age_at_race",
    "rider_bmi",
    "wind_stability_index",
    "weather_temp_mean",
    "weather_temp_trend",
    "weather_rain_prob_mean",
    "weather_precipitation_mean",
    "weather_humidity_mean",
    "gradient_final_km",
    "lag_rider_points_season",
    "lag_rider_rank_season",
    "lag_race_competitiveness_median",
    "lag_team_power_index",
]

TARGET_CONFIGS = {
    "top5": {
        "label": "Top5",
        "train_file": "y_top5_train.pkl",
        "valid_file": "y_top5_valid.pkl",
        "test_file": "y_top5_test.pkl",
        "score_col": "p_top5_raw",
        "actual_col": "actual_top5",
    },
    "top10": {
        "label": "Top10",
        "train_file": "y_top10_train.pkl",
        "valid_file": "y_top10_valid.pkl",
        "test_file": "y_top10_test.pkl",
        "score_col": "p_top10_raw",
        "actual_col": "actual_top10",
    },
    "top20": {
        "label": "Top20",
        "train_file": "y_top20_train.pkl",
        "valid_file": "y_top20_valid.pkl",
        "test_file": "y_top20_test.pkl",
        "score_col": "p_top20_raw",
        "actual_col": "actual_top20",
    },
}

EXPECTED_SHAPES = {
    "train": (169349, 17),
    "valid": (8897, 17),
    "test": (17802, 17),
}


In [3]:
RUN_API = True
FORCE_RERUN = False
SEED = 42
TABPFN_CLIENT_N_ESTIMATORS_MAX = 8

TUNING_DIR = TABPFN_DIR / "02_tuning_top10"
RESULT_DIR = TABPFN_DIR / "03_final_three_targets"
PREDICTION_ROOT = RESULT_DIR / "predictions"
CASE_STUDY_DIR = RESULT_DIR / "case_studies"
THINKING_DIR = RESULT_DIR / "thinking_comparison"
for path in [RESULT_DIR, PREDICTION_ROOT, CASE_STUDY_DIR, THINKING_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def seconds_since(start):
    return round(perf_counter() - start, 3)


def safe_json_dumps(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=True, default=str)


def sanitize_label(value):
    keep = []
    for char in str(value):
        keep.append(char if char.isalnum() or char in "-_" else "_")
    return "".join(keep).strip("_")


## Finale Konfiguration laden


In [4]:
selected_path = TUNING_DIR / "tabpfn_selected_classifier_params.json"
if selected_path.exists():
    with open(selected_path) as f:
        selected = json.load(f)
else:
    selected = {
        "model_family": "TabPFNClassifier",
        "selection_target": "top10",
        "selection_metric": "roc_auc",
        "selection_validation_context": "X_valid, meta_year == 2023",
        "params": {"n_estimators": 4},
        "excluded_from_validation": ["thinking_effort", "thinking_metric", "thinking_mode"],
        "selection_reason": "Fallback placeholder because selected JSON was not found. Run 12-02 first for a real selection.",
    }
    warnings.warn(f"{selected_path} fehlt. Nutze Fallback-Parameter {selected['params']} fuer Notebook-Aufbau/Caches.")

assert selected["model_family"] == "TabPFNClassifier"
assert selected["selection_target"] == "top10"
assert selected["selection_metric"] == "roc_auc"
assert selected["selection_validation_context"] == "X_valid, meta_year == 2023"
assert set(selected["excluded_from_validation"]) == {"thinking_mode", "thinking_effort", "thinking_metric"}
selected


{'model_family': 'TabPFNClassifier',
 'selection_target': 'top10',
 'selection_metric': 'roc_auc',
 'selection_train_context': 'X_train, meta_year <= 2022',
 'selection_validation_context': 'X_valid, meta_year == 2023',
 'selection_train_rows': 169349,
 'validation_rows': 8897,
 'params': {'average_before_softmax': True,
  'balance_probabilities': True,
  'n_estimators': 4,
  'softmax_temperature': 1.0},
 'excluded_from_validation': ['thinking_effort',
  'thinking_metric',
  'thinking_mode'],
 'validation_metrics': {'roc_auc': 0.7831280194000072,
  'average_precision': 0.27404284799510403},
 'validation_runtime': {'fit_seconds': 0.434,
  'predict_seconds': 23.027,
  'total_seconds': 23.496},
 'selection_reason': 'Best validation roc_auc; runtime used as tie-breaker.'}

## Daten laden


In [5]:
def read_pickle(name):
    path = MODEL_DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_pickle(path)

X_train = read_pickle("X_train.pkl")
X_valid = read_pickle("X_valid.pkl")
X_test = read_pickle("X_test.pkl")
X_train_final = pd.concat([X_train, X_valid], axis=0)

y_rank_test = read_pickle("y_rank_test.pkl")
groups_test = read_pickle("groups_test.pkl")
meta_test = read_pickle("meta_test.pkl")

y_targets = {}
for target, cfg in TARGET_CONFIGS.items():
    y_train = read_pickle(cfg["train_file"])
    y_valid = read_pickle(cfg["valid_file"])
    y_test = read_pickle(cfg["test_file"])
    y_targets[(target, "train_final")] = pd.concat([y_train, y_valid], axis=0)
    y_targets[(target, "test")] = y_test

assert list(X_train_final.columns) == FEATURE_COLUMNS
assert list(X_test.columns) == FEATURE_COLUMNS
if X_train_final.shape != (178246, 17):
    warnings.warn(f"X_train_final Shape weicht ab: {X_train_final.shape}")
if X_test.shape != EXPECTED_SHAPES["test"]:
    warnings.warn(f"X_test Shape weicht ab: {X_test.shape}")
assert set(meta_test["meta_year"].unique()).issubset({2024, 2025})
assert meta_test["stage_id"].nunique() == 112

print("X_train_final", X_train_final.shape)
print("X_test", X_test.shape)


X_train_final (178246, 17)
X_test (17802, 17)


## TabPFN-Client und finale Varianten definieren


In [6]:
TabPFNClassifier = None
client_import_error = None
try:
    from tabpfn_client import TabPFNClassifier as _TabPFNClassifier
    TabPFNClassifier = _TabPFNClassifier
    client_source = "tabpfn_client"
except Exception as exc_client:
    try:
        from tabpfn import TabPFNClassifier as _TabPFNClassifier
        TabPFNClassifier = _TabPFNClassifier
        client_source = "tabpfn"
    except Exception as exc_local:
        client_import_error = f"tabpfn_client: {exc_client}; tabpfn: {exc_local}"
        client_source = None

if TabPFNClassifier is None:
    warnings.warn(f"Kein TabPFNClassifier importierbar. RUN_API-Laeufe bleiben deaktiviert. {client_import_error}")
    supported_params = {}
else:
    try:
        supported_params = TabPFNClassifier().get_params()
    except Exception as exc:
        warnings.warn(f"TabPFNClassifier konnte nicht instanziiert werden: {exc}")
        supported_params = {}

FINAL_TEST_VARIANTS = [
    {
        "variant_name": "standard",
        "extra_params": {},
        "role": "selected_params_without_thinking",
    },
    {
        "variant_name": "thinking_medium_roc_auc",
        "extra_params": {
            "thinking_mode": True,
            "thinking_effort": "medium",
            "thinking_metric": "roc_auc",
        },
        "role": "final_test_comparison_only",
    },
]

variant_support_rows = []
for variant in FINAL_TEST_VARIANTS:
    extra = variant["extra_params"]
    unsupported = [key for key in extra if key not in supported_params]
    can_run = (variant["variant_name"] == "standard") or (not unsupported)
    variant_support_rows.append({
        "variant_name": variant["variant_name"],
        "role": variant["role"],
        "extra_params_json": safe_json_dumps(extra),
        "unsupported_params": ", ".join(unsupported),
        "can_run": can_run,
        "client_source": client_source,
    })

variant_support = pd.DataFrame(variant_support_rows)
variant_support.to_csv(THINKING_DIR / "tabpfn_thinking_variant_support.csv", index=False)
variant_support


,variant_name,role,extra_params_json,unsupported_params,can_run,client_source
0,standard,selected_params_without_thinking,{},,True,tabpfn_client
1,thinking_medium_roc_auc,final_test_comparison_only,"{""thinking_effort"": ""medium"", ""thinking_metric...",,True,tabpfn_client


## Prediction- und Evaluationsfunktionen


In [7]:
def final_api_limit_error(params):
    if client_source == "tabpfn_client" and "n_estimators" in params:
        try:
            n_estimators = int(params["n_estimators"])
        except (TypeError, ValueError):
            return f"n_estimators ist nicht als int interpretierbar: {params['n_estimators']}"
        if n_estimators > TABPFN_CLIENT_N_ESTIMATORS_MAX:
            return (
                "tabpfn_client/API-Limit: n_estimators muss <= "
                f"{TABPFN_CLIENT_N_ESTIMATORS_MAX} sein, erhalten: {n_estimators}"
            )
    return None


def filter_supported_final_params(candidate_params, supported_params):
    filtered = {k: v for k, v in candidate_params.items() if k in supported_params}
    limit_error = final_api_limit_error(filtered)
    if limit_error is not None:
        warnings.warn(f"{limit_error}. Setze n_estimators fuer diesen API-Lauf auf {TABPFN_CLIENT_N_ESTIMATORS_MAX}.")
        filtered["n_estimators"] = TABPFN_CLIENT_N_ESTIMATORS_MAX
    return filtered


def positive_class_probability(model, X):
    proba = model.predict_proba(X)
    classes = list(model.classes_)
    if 1 not in classes:
        raise ValueError(f"Positive Klasse 1 fehlt in model.classes_: {classes}")
    positive_idx = classes.index(1)
    return proba[:, positive_idx]


def make_actual_relevance_graded(actual_rank):
    rank = pd.to_numeric(actual_rank, errors="coerce")
    return np.select(
        [
            rank.between(1, 5, inclusive="both"),
            rank.between(6, 10, inclusive="both"),
            rank.between(11, 20, inclusive="both"),
        ],
        [3, 2, 1],
        default=0,
    ).astype(int)


def build_target_prediction_df(target, score_values, variant_name, params):
    cfg = TARGET_CONFIGS[target]
    pred = meta_test.reset_index(drop=True).copy()
    pred["stage_id"] = pd.Series(groups_test).reset_index(drop=True).astype(str).values
    pred["actual_rank"] = pd.Series(y_rank_test).reset_index(drop=True).values
    pred[cfg["actual_col"]] = pd.Series(y_targets[(target, "test")]).reset_index(drop=True).astype(int).values
    pred[cfg["score_col"]] = score_values
    pred["target"] = target
    pred["variant_name"] = variant_name
    for key, value in params.items():
        pred[key] = value
    return pred


def run_or_load_target_predictions(variant, target):
    variant_name = variant["variant_name"]
    variant_dir = PREDICTION_ROOT / sanitize_label(variant_name)
    variant_dir.mkdir(parents=True, exist_ok=True)
    pred_path = variant_dir / f"tabpfn_final_predictions_{target}_2024_2025.csv"
    run_start = perf_counter()
    runtime = {
        "variant_name": variant_name,
        "target": target,
        "runtime_status": None,
        "fit_seconds": np.nan,
        "predict_seconds": np.nan,
        "total_seconds": np.nan,
        "cache_load_seconds": np.nan,
        "train_rows": len(X_train_final),
        "test_rows": len(X_test),
        "prediction_path": str(pred_path),
        "error_message": None,
    }

    if pred_path.exists() and not FORCE_RERUN:
        cache_start = perf_counter()
        pred = pd.read_csv(pred_path)
        runtime["cache_load_seconds"] = seconds_since(cache_start)
        runtime["runtime_status"] = "cache_hit"
        runtime["total_seconds"] = seconds_since(run_start)
        return pred, runtime

    support = variant_support.loc[variant_support["variant_name"] == variant_name].iloc[0]
    if not bool(support["can_run"]):
        runtime["runtime_status"] = "unsupported_variant_params"
        runtime["total_seconds"] = seconds_since(run_start)
        return None, runtime

    if not RUN_API:
        runtime["runtime_status"] = "skipped_missing_cache"
        runtime["total_seconds"] = seconds_since(run_start)
        return None, runtime

    if TabPFNClassifier is None:
        runtime["runtime_status"] = "missing_classifier"
        runtime["total_seconds"] = seconds_since(run_start)
        return None, runtime

    selected_params = selected.get("params", {})
    params = filter_supported_final_params({**selected_params, **variant["extra_params"]}, supported_params)

    try:
        fit_start = perf_counter()
        model = TabPFNClassifier(**params)
        model.fit(X_train_final, y_targets[(target, "train_final")])
        runtime["fit_seconds"] = seconds_since(fit_start)

        predict_start = perf_counter()
        p_positive = positive_class_probability(model, X_test)
        runtime["predict_seconds"] = seconds_since(predict_start)

        pred = build_target_prediction_df(target, p_positive, variant_name, params)
        pred.to_csv(pred_path, index=False)
        runtime["runtime_status"] = "api_run"
        runtime["total_seconds"] = seconds_since(run_start)
        return pred, runtime
    except Exception as exc:
        runtime["runtime_status"] = "api_error"
        runtime["error_message"] = repr(exc)
        runtime["total_seconds"] = seconds_since(run_start)
        return None, runtime


def merge_three_target_predictions(top5_df, top10_df, top20_df):
    key_cols = ["variant_name", "meta_year", "meta_race", "stage_nr", "stage_id", "meta_name", "meta_current_team", "actual_rank"]
    base = top5_df[key_cols + ["actual_top5", "p_top5_raw"]].copy()
    top10_part = top10_df[["variant_name", "stage_id", "meta_name", "actual_top10", "p_top10_raw"]].copy()
    top20_part = top20_df[["variant_name", "stage_id", "meta_name", "actual_top20", "p_top20_raw"]].copy()
    out = base.merge(top10_part, on=["variant_name", "stage_id", "meta_name"], how="inner")
    out = out.merge(top20_part, on=["variant_name", "stage_id", "meta_name"], how="inner")
    out["actual_relevance_graded"] = make_actual_relevance_graded(out["actual_rank"])
    return out


def build_class_probabilities(pred_df):
    out = pred_df.copy()
    out["p_class_top5"] = out["p_top5_raw"]
    out["p_class_6_10"] = out["p_top10_raw"] - out["p_top5_raw"]
    out["p_class_11_20"] = out["p_top20_raw"] - out["p_top10_raw"]
    out["p_class_outside_top20"] = 1 - out["p_top20_raw"]
    class_cols = ["p_class_top5", "p_class_6_10", "p_class_11_20", "p_class_outside_top20"]
    out["has_negative_class_probability"] = (out[class_cols] < -1e-9).any(axis=1)
    return out


def assign_most_likely_disjoint_class(pred_df):
    out = pred_df.copy()
    class_cols = ["p_class_top5", "p_class_6_10", "p_class_11_20", "p_class_outside_top20"]
    labels = {
        "p_class_top5": "top5",
        "p_class_6_10": "rank_6_10",
        "p_class_11_20": "rank_11_20",
        "p_class_outside_top20": "outside_top20",
    }
    out["most_likely_class_col"] = out[class_cols].idxmax(axis=1)
    out["most_likely_disjoint_class"] = out["most_likely_class_col"].map(labels)
    out["most_likely_disjoint_class_probability"] = out[class_cols].max(axis=1)
    return out


def build_topk_raw_sum_score(pred_df):
    out = pred_df.copy()
    out["score_topk_raw_sum"] = out["p_top5_raw"] + out["p_top10_raw"] + out["p_top20_raw"]
    return out


def rank_topk_raw_sum_by_stage(pred_df):
    out = pred_df.copy()
    out["rank_topk_raw_sum"] = out.groupby("stage_id")["score_topk_raw_sum"].rank(method="first", ascending=False).astype(int)
    return out


def dcg_at_k(relevance, k):
    rel = np.asarray(relevance, dtype=float)[:k]
    if rel.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, rel.size + 2))
    return float(np.sum((2 ** rel - 1) / discounts))


def ndcg_for_stage(stage_df, k):
    ordered = stage_df.sort_values("score_topk_raw_sum", ascending=False)
    actual = stage_df.sort_values("actual_relevance_graded", ascending=False)
    denom = dcg_at_k(actual["actual_relevance_graded"].to_numpy(), k)
    if denom == 0:
        return np.nan
    return dcg_at_k(ordered["actual_relevance_graded"].to_numpy(), k) / denom


def top_n_winner_accuracy_for_stage(stage_df, n):
    winners = stage_df[stage_df["actual_rank"] == 1]
    if winners.empty:
        return np.nan
    return bool((winners["rank_topk_raw_sum"] <= n).any())


def average_precision_for_stage(stage_df):
    ordered = stage_df.sort_values("score_topk_raw_sum", ascending=False).reset_index(drop=True)
    relevant = ordered["actual_top20"].astype(int).to_numpy()
    n_relevant = int(relevant.sum())
    if n_relevant == 0:
        return np.nan
    precisions = []
    hits = 0
    for idx, is_relevant in enumerate(relevant, start=1):
        if is_relevant:
            hits += 1
            precisions.append(hits / idx)
    return float(np.sum(precisions) / n_relevant)


def evaluate_stage_metrics(pred_df):
    rows = []
    for stage_id, stage in pred_df.groupby("stage_id"):
        row = {
            "variant_name": stage["variant_name"].iloc[0],
            "score_name": "score_topk_raw_sum",
            "stage_id": stage_id,
            "meta_year": stage["meta_year"].iloc[0],
            "meta_race": stage["meta_race"].iloc[0],
            "stage_nr": stage["stage_nr"].iloc[0],
            "rows": len(stage),
        }
        for k in [5, 10, 20]:
            row[f"ndcg_at_{k}"] = ndcg_for_stage(stage, k)
        for n in [1, 5, 10, 20]:
            row[f"top_{n}_accuracy"] = top_n_winner_accuracy_for_stage(stage, n)
        row["map"] = average_precision_for_stage(stage)
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_metrics(stage_metrics, test_scope="overall"):
    if stage_metrics.empty:
        return pd.DataFrame()
    metric_cols = ["ndcg_at_5", "ndcg_at_10", "ndcg_at_20", "top_1_accuracy", "top_5_accuracy", "top_10_accuracy", "top_20_accuracy", "map"]
    row = {
        "variant_name": stage_metrics["variant_name"].iloc[0],
        "score_name": "score_topk_raw_sum",
        "test_scope": test_scope,
        "n_stages": stage_metrics["stage_id"].nunique(),
    }
    for col in metric_cols:
        row[col] = stage_metrics[col].astype(float).mean()
    return pd.DataFrame([row])


## Finale Prediction-Laeufe ausfuehren oder aus Cache laden


In [8]:
runtime_rows = []
combined_predictions = []
variant_target_predictions = {}

for variant in FINAL_TEST_VARIANTS:
    variant_name = variant["variant_name"]
    target_preds = {}
    for target in TARGET_CONFIGS:
        print(f"Variant={variant_name}, target={target}")
        pred, runtime = run_or_load_target_predictions(variant, target)
        runtime_rows.append(runtime)
        if pred is not None:
            target_preds[target] = pred
            variant_target_predictions[(variant_name, target)] = pred
    if set(target_preds) == set(TARGET_CONFIGS):
        combined = merge_three_target_predictions(target_preds["top5"], target_preds["top10"], target_preds["top20"])
        combined = build_class_probabilities(combined)
        combined = assign_most_likely_disjoint_class(combined)
        combined = build_topk_raw_sum_score(combined)
        combined = rank_topk_raw_sum_by_stage(combined)
        variant_dir = PREDICTION_ROOT / sanitize_label(variant_name)
        combined.to_csv(variant_dir / "tabpfn_final_predictions_2024_2025_all_targets.csv", index=False)
        combined_predictions.append(combined)

runtime_df = pd.DataFrame(runtime_rows)
runtime_df.to_csv(RESULT_DIR / "tabpfn_final_runtime_by_target.csv", index=False)
runtime_df.to_csv(RESULT_DIR / "tabpfn_final_runtime.csv", index=False)

runtime_by_variant = (
    runtime_df.groupby("variant_name", as_index=False)
    .agg(
        total_fit_seconds=("fit_seconds", "sum"),
        total_predict_seconds=("predict_seconds", "sum"),
        total_runtime_seconds=("total_seconds", "sum"),
        mean_runtime_seconds_per_target=("total_seconds", "mean"),
        targets_completed=("runtime_status", lambda s: int(s.isin(["api_run", "cache_hit"]).sum())),
        runtime_status=("runtime_status", lambda s: ", ".join(sorted(set(map(str, s))))),
    )
)
runtime_by_variant.to_csv(RESULT_DIR / "tabpfn_final_runtime_by_variant.csv", index=False)
runtime_by_variant.to_csv(THINKING_DIR / "tabpfn_thinking_runtime.csv", index=False)

runtime_df


Variant=standard, target=top5


Found existing access token, reusing it for authentication.

00:06 Fitting... Done!
00:40 Predicting... Done!
Variant=standard, target=top10
00:06 Fitting... Done!
00:35 Predicting... Done!
Variant=standard, target=top20
00:17 Fitting... Done!
00:36 Predicting... Done!
Variant=thinking_medium_roc_auc, target=top5
00:00 Fitting... -

The provided train set hashes match previously uploaded train sets.


43:14 Fitting... Done!
00:00 Predicting... -

...

The provided train set hashes match previously uploaded train sets.


41:32 Fitting... Done!
00:00 Predicting... -

The provided test set hash matches a previously uploaded test set.


00:31 Predicting... Done!


,variant_name,target,runtime_status,fit_seconds,predict_seconds,total_seconds,cache_load_seconds,train_rows,test_rows,prediction_path,error_message
0,standard,top5,api_run,7.352,41.112,48.535,NaN,178246,17802,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,NaN
1,standard,top10,api_run,6.556,35.735,42.353,NaN,178246,17802,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,NaN
2,standard,top20,api_run,17.999,36.965,55.025,NaN,178246,17802,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,NaN
3,thinking_medium_roc_auc,top5,api_run,2594.801,39.017,2633.903,NaN,178246,17802,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,NaN
4,thinking_medium_roc_auc,top10,api_error,NaN,NaN,2610.450,NaN,178246,17802,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,ReadTimeout('The read operation timed out')
5,thinking_medium_roc_auc,top20,api_run,2493.236,31.557,2524.904,NaN,178246,17802,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,NaN


## Finale Metriken, Klassenverteilung und Thinking-Vergleich


In [9]:
all_stage_metrics = []
overall_metrics = []
by_year_metrics = []
class_distribution_rows = []

for combined in combined_predictions:
    stage_metrics = evaluate_stage_metrics(combined)
    all_stage_metrics.append(stage_metrics)
    overall_metrics.append(summarize_metrics(stage_metrics, "overall"))
    for year, part in stage_metrics.groupby("meta_year"):
        by_year_metrics.append(summarize_metrics(part, str(year)))
    class_dist = (
        combined.groupby(["variant_name", "most_likely_disjoint_class"], dropna=False)
        .size()
        .reset_index(name="rows")
    )
    class_dist["scope"] = "overall"
    class_distribution_rows.append(class_dist)
    class_dist_year = (
        combined.groupby(["variant_name", "meta_year", "most_likely_disjoint_class"], dropna=False)
        .size()
        .reset_index(name="rows")
        .rename(columns={"meta_year": "scope"})
    )
    class_distribution_rows.append(class_dist_year)

stage_metrics_df = pd.concat(all_stage_metrics, ignore_index=True) if all_stage_metrics else pd.DataFrame()
overall_metrics_df = pd.concat(overall_metrics, ignore_index=True) if overall_metrics else pd.DataFrame()
by_year_metrics_df = pd.concat(by_year_metrics, ignore_index=True) if by_year_metrics else pd.DataFrame()
class_distribution_df = pd.concat(class_distribution_rows, ignore_index=True) if class_distribution_rows else pd.DataFrame()

stage_metrics_df.to_csv(RESULT_DIR / "tabpfn_final_stage_metrics.csv", index=False)
overall_metrics_df.to_csv(RESULT_DIR / "tabpfn_final_metrics_overall.csv", index=False)
by_year_metrics_df.to_csv(RESULT_DIR / "tabpfn_final_metrics_by_year.csv", index=False)
class_distribution_df.to_csv(RESULT_DIR / "tabpfn_final_disjoint_class_distribution.csv", index=False)

thinking_comparison = overall_metrics_df.merge(runtime_by_variant, on="variant_name", how="left") if not overall_metrics_df.empty else pd.DataFrame()
thinking_comparison.to_csv(RESULT_DIR / "tabpfn_final_thinking_comparison.csv", index=False)
thinking_comparison.to_csv(THINKING_DIR / "tabpfn_thinking_metrics.csv", index=False)

usage_df = pd.DataFrame([{"variant_name": row["variant_name"], "usage_status": "not_captured_in_notebook", "note": "Token/credit usage must be filled from client-specific usage APIs if available."} for _, row in runtime_by_variant.iterrows()])
usage_df.to_csv(RESULT_DIR / "tabpfn_final_usage.csv", index=False)
usage_df.to_csv(THINKING_DIR / "tabpfn_thinking_usage.csv", index=False)

overall_metrics_df


,variant_name,score_name,test_scope,n_stages,ndcg_at_5,ndcg_at_10,ndcg_at_20,top_1_accuracy,top_5_accuracy,top_10_accuracy,top_20_accuracy,map
0,standard,score_topk_raw_sum,overall,112,0.356543,0.375069,0.430545,0.207207,0.414414,0.576577,0.666667,0.401186


## Case Study: Tour de France 2025, Stage 12 und 16


In [10]:
def build_case_study(pred_df):
    mask = (
        (pred_df["meta_race"].astype(str) == "tour-de-france")
        & (pred_df["meta_year"].astype(int) == 2025)
        & (pred_df["stage_nr"].astype(int).isin([12, 16]))
    )
    cases = pred_df.loc[mask].copy()
    rows = []
    if cases.empty:
        return pd.DataFrame()
    for (variant_name, stage_id), stage in cases.groupby(["variant_name", "stage_id"]):
        predicted = stage.sort_values("score_topk_raw_sum", ascending=False).head(20).copy()
        predicted["view"] = "predicted_top20_by_score_topk_raw_sum"
        predicted["position"] = range(1, len(predicted) + 1)
        actual = stage.sort_values("actual_rank", ascending=True).head(20).copy()
        actual["view"] = "actual_top20"
        actual["position"] = range(1, len(actual) + 1)
        rows.extend([predicted, actual])
    out = pd.concat(rows, ignore_index=True)
    out = out.rename(columns={"meta_name": "rider", "meta_current_team": "team", "rank_topk_raw_sum": "predicted_rank"})
    keep_cols = [
        "stage_id", "stage_nr", "variant_name", "view", "position", "rider", "team",
        "actual_rank", "predicted_rank", "score_topk_raw_sum",
        "most_likely_disjoint_class", "most_likely_disjoint_class_probability",
        "p_top5_raw", "p_top10_raw", "p_top20_raw",
        "p_class_top5", "p_class_6_10", "p_class_11_20", "p_class_outside_top20",
        "actual_top5", "actual_top10", "actual_top20",
    ]
    return out[keep_cols]

case_studies = []
for combined in combined_predictions:
    case = build_case_study(combined)
    if not case.empty:
        case_studies.append(case)

case_study_df = pd.concat(case_studies, ignore_index=True) if case_studies else pd.DataFrame()
case_study_df.to_csv(CASE_STUDY_DIR / "tabpfn_case_study_tdf2025_stage12_16.csv", index=False)

summary_path = CASE_STUDY_DIR / "tabpfn_case_study_tdf2025_stage12_16_summary.md"
with open(summary_path, "w") as f:
    f.write("# TabPFN Case Study TDF 2025 Stage 12 und 16\n\n")
    if case_study_df.empty:
        f.write("Keine Case-Study-Predictions verfuegbar. RUN_API=True setzen oder Prediction-Caches bereitstellen.\n")
    else:
        for (variant, stage_id), part in case_study_df.groupby(["variant_name", "stage_id"]):
            f.write(f"## {variant} - {stage_id}\n\n")
            f.write(f"Zeilen in Case Study: {len(part)}\n\n")
            f.write("Bitte fachlich kommentieren: Treffer Top5/Top10/Top20, Ueberschaetzungen, Unterschaetzungen, wahrscheinlichste disjunkte Klassen und Thinking-Laufzeit.\n\n")

case_study_df.head(40)


,stage_id,stage_nr,variant_name,view,position,rider,team,actual_rank,predicted_rank,score_topk_raw_sum,most_likely_disjoint_class,most_likely_disjoint_class_probability,p_top5_raw,p_top10_raw,p_top20_raw,p_class_top5,p_class_6_10,p_class_11_20,p_class_outside_top20,actual_top5,actual_top10,actual_top20
0,tour-de-france_2025_ST12,12,standard,predicted_top20_by_score_topk_raw_sum,1,Tadej Pogačar,UAE Team Emirates - XRG,1.0,1,2.866931,top5,0.963822,0.963822,0.949622,0.953486,0.963822,-0.014200,0.003864,0.046514,1,1,1
1,tour-de-france_2025_ST12,12,standard,predicted_top20_by_score_topk_raw_sum,2,Jonas Vingegaard,Team Visma | Lease a Bike,2.0,2,2.854630,top5,0.940333,0.940333,0.941334,0.972962,0.940333,0.001001,0.031628,0.027038,1,1,1
2,tour-de-france_2025_ST12,12,standard,predicted_top20_by_score_topk_raw_sum,3,Aleksandr Vlasov,Red Bull - BORA - hansgrohe,34.0,3,2.813523,top5,0.892482,0.892482,0.952180,0.968861,0.892482,0.059698,0.016681,0.031139,0,0,0
3,tour-de-france_2025_ST12,12,standard,predicted_top20_by_score_topk_raw_sum,4,Matteo Jorgenson,Team Visma | Lease a Bike,15.0,4,2.781375,top5,0.933063,0.933063,0.930711,0.917601,0.933063,-0.002353,-0.013110,0.082399,0,0,1
4,tour-de-france_2025_ST12,12,standard,predicted_top20_by_score_topk_raw_sum,5,Ben O'Connor,Team Jayco AlUla,16.0,5,2.770342,top5,0.912284,0.912284,0.914930,0.943128,0.912284,0.002645,0.028199,0.056872,0,0,1
5,tour-de-france_2025_ST12,12,standard,predicted_top20_by_score_topk_raw_sum,6,Marc Hirschi,Tudor Pro Cycling Team,57.0,6,2.735316,top5,0.935072,0.935072,0.902277,0.897967,0.935072,-0.032794,-0.004310,0.102033,0,0,0
6,tour-de-france_2025_ST12,12,standard,predicted_top20_by_score_topk_raw_sum,7,Adam Yates,UAE Team Emirates - XRG,23.0,7,2.725707,top5,0.863045,0.863045,0.910945,0.951718,0.863045,0.047900,0.040773,0.048282,0,0,0
7,tour-de-france_2025_ST12,12,standard,predicted_top20_by_score_topk_raw_sum,8,Remco Evenepoel,Soudal Quick-Step,7.0,8,2.718227,top5,0.944047,0.944047,0.891660,0.882519,0.944047,-0.052386,-0.009141,0.117481,0,1,1
8,tour-de-france_2025_ST12,12,standard,predicted_top20_by_score_topk_raw_sum,9,Enric Mas,Movistar Team,33.0,9,2.710521,top5,0.815411,0.815411,0.925384,0.969727,0.815411,0.109973,0.044343,0.030273,0,0,0
9,tour-de-france_2025_ST12,12,standard,predicted_top20_by_score_topk_raw_sum,10,Carlos Rodríguez,INEOS Grenadiers,22.0,10,2.708468,top5,0.854001,0.854001,0.902444,0.952022,0.854001,0.048444,0.049578,0.047978,0,0,0
